# Telco Customer Churn Analysis — Exploratory Data Analysis

## 1. Objective & Analytical Scope

Explore customer churn patterns across customer lifecycle, contract, \
service, payment, and spending characteristics.

The analysis will identify associations and patterns that can be investigated \
further during feature analysis.

**Analytical grain:** One row = one customer.

## 2. Setup & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/processed/telco_cleaned.csv")

## 3. Dataset Overview

The analysis uses the cleaned dataset produced during Phase 1.

| Attribute | Value |
|---|---|
| Rows | 7,043 |
| Columns | 21 |
| Target variable | `Churn` |

In [ ]:
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()
df.info()

## 4. Overall Churn Distribution

**Business question:** What proportion of customers churned in this dataset?

In [ ]:
churn_counts = df["Churn"].value_counts()

churn_rate = (
    df["Churn"].eq("Yes").mean() * 100
)

print(f"Churned customers:  {churn_counts['Yes']:,}")
print(f"Retained customers: {churn_counts['No']:,}")
print(f"Overall churn rate: {churn_rate:.2f}%")

In [ ]:
churn_counts = df["Churn"].value_counts()

plt.figure(figsize=(7, 5))

plt.bar(
    churn_counts.index,
    churn_counts.values
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")

for i, value in enumerate(churn_counts.values):
    plt.text(
        i,
        value,
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

### Insight — Overall Churn

The dataset contains **7,043 customers**, of whom **1,869 (26.54%)** churned. \
The approximate 3:1 retained-to-churned ratio represents a class imbalance \
that should be considered when interpreting subsequent churn analysis.

## 5. Churn by Contract Type

**Business question:** How does churn vary by contract type?

In [ ]:
contract_churn_rate = (
    df.groupby("Contract")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .sort_values(ascending=False)
      .round(2)
)

contract_churn_rate

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    contract_churn_rate.index,
    contract_churn_rate.values
)

plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")

for i, value in enumerate(contract_churn_rate.values):
    plt.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

### Insight — Contract Type

Month-to-month customers show a **42.71%** churn rate, compared with **11.27%** \
for one-year contracts and **2.83%** for two-year contracts — a \
**39.88 percentage-point gap** between the shortest and longest contract types.

Contract structure shows one of the strongest observed associations with churn \
in this dataset and warrants deeper investigation in the feature-analysis phase.

## 6. Churn by Tenure

**Business question:** How does churn vary across different stages of the customer lifecycle?

In [ ]:
tenure_bins = [0, 12, 24, 48, 72]
tenure_labels = [
    "0–12 months",
    "13–24 months",
    "25–48 months",
    "49–72 months"
]

df["tenure_band_eda"] = pd.cut(
    df["tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    include_lowest=True
)

tenure_churn_rate = (
    df.groupby("tenure_band_eda", observed=True)["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .sort_values(ascending=False)
      .round(2)
)

tenure_churn_rate

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    tenure_churn_rate.index.astype(str),
    tenure_churn_rate.values
)

plt.title("Churn Rate by Tenure Band")
plt.xlabel("Tenure")
plt.ylabel("Churn Rate (%)")

for i, value in enumerate(tenure_churn_rate.values):
    plt.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

### Insight — Tenure

Customers with **0–12 months** of tenure show a **47.44%** churn rate, \
compared with **9.51%** among customers with **49–72 months** of tenure \
— a **37.93 percentage-point difference**.

The relationship is monotonically decreasing: churn rate falls at each \
successive tenure band. This identifies the early customer lifecycle as a \
priority segment for further retention analysis.

## 7. Churn by Internet Service

**Business question:** How does churn vary by internet service type?

In [ ]:
internet_churn_rate = (
    df.groupby("InternetService")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .sort_values(ascending=False)
      .round(2)
)

internet_churn_rate

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    internet_churn_rate.index,
    internet_churn_rate.values
)

plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Churn Rate (%)")

for i, value in enumerate(internet_churn_rate.values):
    plt.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

### Insight — Internet Service

Fiber-optic customers show a **41.89%** churn rate, compared with **18.96%** \
for DSL customers and **7.40%** for customers without internet service \
— a **34.49 percentage-point gap** between fiber-optic and no-internet customers.

The elevated churn among fiber-optic customers warrants further investigation \
alongside contract type, tenure, and other service characteristics.

## 8. Churn by Payment Method

**Business question:** How does churn vary by payment method?

In [ ]:
payment_churn_rate = (
    df.groupby("PaymentMethod")["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .sort_values(ascending=False)
      .round(2)
)

payment_churn_rate

In [ ]:
plt.figure(figsize=(9, 5))

payment_churn_rate_sorted = payment_churn_rate.sort_values()

payment_churn_rate_sorted.plot(
    kind="barh"
)

plt.title("Churn Rate by Payment Method")
plt.xlabel("Churn Rate (%)")
plt.ylabel("Payment Method")

for i, value in enumerate(payment_churn_rate_sorted.values):
    plt.text(
        value,
        i,
        f"{value:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

### Insight — Payment Method

Electronic-check customers show a **45.29%** churn rate — substantially higher \
than the other payment methods, which range from **15.24%** to **19.11%** \
(approximately a **26 percentage-point gap**).

Payment method may act as a proxy for a different underlying customer \
characteristic, such as commitment level or contract type, rather than being \
a direct driver of churn. This warrants cross-tabulation in the \
feature-analysis phase.

## 9. Churn by Monthly Charges

**Business question:** How does churn vary across monthly charge bands?

In [ ]:
charge_bins = [0, 50, 100, float("inf")]

charge_labels = [
    "0–50",
    "50–100",
    "100+"
]

df["charge_band_eda"] = pd.cut(
    df["MonthlyCharges"],
    bins=charge_bins,
    labels=charge_labels,
    include_lowest=True
)

monthly_charge_churn_rate = (
    df.groupby("charge_band_eda", observed=True)["Churn"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .round(2)
)

monthly_charge_churn_rate

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    monthly_charge_churn_rate.index.astype(str),
    monthly_charge_churn_rate.values
)

plt.title("Churn Rate by Monthly Charges")
plt.xlabel("Monthly Charges")
plt.ylabel("Churn Rate (%)")

for i, value in enumerate(monthly_charge_churn_rate.values):
    plt.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

### Insight — Monthly Charges

The **50–100** charge band has the highest observed churn rate at **32.67%**, \
followed by the **100+** band at **28.05%** and the **0–50** band at **15.70%**.

The relationship is not strictly monotonic: the highest charge band (100+) \
shows a lower churn rate than the mid-range band (50–100). This suggests \
that monthly charges alone do not explain churn behaviour and that the \
relationship likely interacts with other variables such as contract type \
and internet service.

## 10. Numeric Variable Distributions

**Business question:** How do the distributions of tenure, MonthlyCharges, and TotalCharges differ between churned and retained customers?

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="tenure",
    hue="Churn",
    bins=30,
    kde=True,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Tenure Distribution by Churn Status")
plt.xlabel("Tenure (Months)")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="MonthlyCharges",
    hue="Churn",
    bins=30,
    kde=True,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Monthly Charges Distribution by Churn Status")
plt.xlabel("Monthly Charges")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="TotalCharges",
    hue="Churn",
    bins=30,
    kde=True,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Total Charges Distribution by Churn Status")
plt.xlabel("Total Charges")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

### Insight — Numeric Distributions

**Tenure:** Churned customers are heavily concentrated in the early months, with the churned distribution peaking at low tenure values and declining sharply. Retained customers show a more uniform distribution across tenure bands, with a secondary concentration at higher tenure values. This confirms the tenure-band finding from Section 6 and indicates that the churn risk is highest — and most acute — very early in the customer lifecycle.

**Monthly Charges:** Churned customers show a higher concentration at elevated monthly charge values relative to retained customers. Retained customers are proportionally more concentrated in the lower charge ranges. This is consistent with the charge-band finding in Section 9.

**Total Charges:** Churned customers are heavily concentrated at low total-charge values. This pattern is closely related to tenure because TotalCharges accumulates over the customer's relationship with the company, so it should not be interpreted as an independent churn signal.

## 11. Contract × Internet Service

**Business question:** How does the distribution of internet service vary across contract types, and does this overlap help explain the independently observed high churn rates for both characteristics?

In [ ]:
contract_internet = pd.crosstab(
    df["Contract"],
    df["InternetService"],
    normalize="index"
) * 100

contract_internet.round(2)

In [ ]:
contract_internet.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)

plt.title("Internet Service Mix by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Customer Distribution (%)")
plt.legend(title="Internet Service", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Insight — Contract × Internet Service

The cross-tabulation shows that fiber-optic customers are more concentrated in month-to-month contracts than in longer-term contracts, indicating an important overlap between two high-churn segments.

This overlap should be investigated further in Phase 3 to determine whether the elevated churn associated with fiber-optic service persists after accounting for contract type.

This finding should not be interpreted causally. It identifies a compositional overlap that warrants controlled analysis in the feature-analysis phase.

### Cleanup — Temporary EDA Columns

The tenure and charge band columns were created for exploratory grouping only and are not intended to persist into downstream phases.

In [ ]:
df.drop(
    columns=["tenure_band_eda", "charge_band_eda"],
    inplace=True
)

df.shape  # Should return (7043, 21)

## 12. Numeric Variable Relationships

**Business question:** How do the numeric variables — tenure, MonthlyCharges, \
and TotalCharges — relate to one another?

In [ ]:
numeric_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix — Numeric Variables")
plt.tight_layout()
plt.show()

### Insight — Numeric Relationships

`tenure` and `TotalCharges` show a strong positive correlation of **0.83**. \
This relationship is structurally expected: `TotalCharges` accumulates over \
the customer's relationship with the company, so longer-tenure customers will \
naturally have higher cumulative charges.

`MonthlyCharges` and `TotalCharges` show a moderate positive correlation of **0.65**, \
while `tenure` and `MonthlyCharges` have a weaker positive correlation of **0.25** \
— indicating that longer-tenure customers do not necessarily carry \
significantly higher monthly charges.

> **Note:** Correlation measures linear association and does not establish causation.

## 13. Key EDA Findings

1. **Contract:** Month-to-month customers show a **42.71%** churn rate, compared with \
   **2.83%** for two-year contracts — a **39.88 percentage-point difference**.

2. **Tenure:** Customers with **0–12 months** of tenure show a **47.44%** churn rate, \
   compared with **9.51%** among customers with **49–72 months** \
   — a **37.93 percentage-point difference**.

3. **Internet Service:** Fiber-optic customers show a **41.89%** churn rate, compared \
   with **18.96%** for DSL and **7.40%** for customers without internet service.

4. **Payment Method:** Electronic-check customers show a **45.29%** churn rate \
   — approximately **26 percentage points** above the other payment methods (15.24%–19.11%).

5. **Monthly Charges:** The **50–100** charge band has the highest observed churn rate \
   at **32.67%**. The relationship is not strictly monotonic — the 100+ band (28.05%) \
   shows a lower rate than the 50–100 band.

6. **Numeric Relationships:** `tenure` and `TotalCharges` have a strong positive \
   correlation of **0.83**, reflecting the structural link between customer age \
   and cumulative spend.

## 14. Analytical Limitations

- EDA findings represent observed associations and do not establish causation.
- `tenure` and `TotalCharges` are structurally related because `TotalCharges` \
  accumulates over the customer's relationship with the company. These variables \
  should not be treated as independent in downstream analysis.
- Monthly-charge and tenure bands are exploratory groupings created for \
  visualisation and should not automatically be treated as final business segments.
- The analysis focuses on seven primary explanatory variables. Other variables in the dataset \
  (gender, partner, dependents, phone service, individual add-on services, \
  paperless billing) were not analysed at this stage and may carry additional signal.
- Section 11 identifies a compositional overlap between contract type and internet \
  service. Further controlled analysis is required to determine whether fiber-optic \
  service carries an independent churn signal after accounting for contract type.

## 15. EDA Conclusion

The exploratory analysis identified substantial differences in observed churn \
across contract type, tenure, internet service, payment method, and monthly \
charge bands.

The strongest observed churn concentrations occur among:

| Segment | Churn Rate |
|---|---|
| Customers with 0–12 months of tenure | 47.44% |
| Electronic-check customers | 45.29% |
| Month-to-month customers | 42.71% |
| Fiber-optic customers | 41.89% |

These patterns indicate several areas for deeper segment-level investigation.

The next phase will examine combinations of customer characteristics to \
determine whether specific customer profiles represent meaningful \
high-risk segments.

These findings describe associations within the dataset and should not be \
interpreted as causal relationships.